In [17]:
import io
import sys
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

import lightning as L
from lightning.pytorch.loggers import WandbLogger
from lightning.pytorch.callbacks import ModelCheckpoint
import wandb

from new_data.data_loader import train_df, val_df
import os
import torchmetrics

In [18]:
sys.path.append("/Users/saratramontana/Documents/test_segmentation_model/external/OvaMTA")

In [19]:
from lib.OvaMTA_seg import TransRaUNet_CLF_xiaorong #TransRaUNet_CLF_xiaorong =backbone (feature extraction)+ decoder (segmentation)+ classification head
from utils.smooth_l1_loss import SmoothL1Loss

In [20]:
class OvaSegWrapperDataset(data.Dataset):
    def __init__(self, df, trainsize=352):
        self.df = df.reset_index(drop=True)
        self.trainsize = trainsize

        self.img_transform = transforms.Compose([
            transforms.Resize((trainsize, trainsize)),
            transforms.ToTensor(),
            transforms.Normalize(
                [0.485, 0.456, 0.406],
                [0.229, 0.224, 0.225]
            )
        ])

        self.gt_transform = transforms.Compose([
            transforms.Resize(
                (trainsize, trainsize),
                interpolation=transforms.InterpolationMode.NEAREST #we have binary mask so we use the NEAREST interp mode, otherwise with bilinear we wouldn't have discrete classes
            ),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(io.BytesIO(row["image"])).convert("RGB")
        gt = Image.open(io.BytesIO(row["mask"])).convert("L")

        image = self.img_transform(image)
        gt_np = np.array(gt)
        gt_bin = (gt_np > 0).astype(np.uint8) #prima di gt_transform devo prima binarizzarla 

        gt = Image.fromarray(gt_bin * 255) #perchè quel 255? Perché * 255? Perché Image.fromarray con valori 0 e 1 crea un’immagine molto “scura”, mentre così hai una vera immagine binaria standard: 0 = nero, 255 = bianco. Poi ToTensor() la riporta in: 0.0 1.0
        gt = self.gt_transform(gt) #gt qui sta tra 0 e 1 perché ToTensor divide per 255

        gt = (gt > 0).float()   # binary ROI 

        label = torch.tensor(row["risk_class"] + 1, dtype=torch.long) 
        

        return image, gt, label

In [21]:
train_dataset = OvaSegWrapperDataset(train_df, trainsize=352)
val_dataset = OvaSegWrapperDataset(val_df, trainsize=352)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

In [22]:
# batch = next(iter(train_loader))
# images, gts, labels = batch

# print(images.shape)
# print(gts.shape)
# print(labels.shape)
# print(torch.unique(gts))
# print(torch.unique(labels))

In [23]:
os.chdir("/Users/saratramontana/Documents/test_segmentation_model/external/OvaMTA")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TransRaUNet_CLF_xiaorong(training=True).to(device)

x = torch.randn(2, 3, 352, 352).to(device) #crea un batch finto
out = model(x)

print(type(out))
print(len(out))

for i, elem in enumerate(out):
    if torch.is_tensor(elem):
        print(i, elem.shape)
    else:
        print(i, type(elem)) #la 5 è il feature vector: non mi serve  

<class 'tuple'>
6
0 torch.Size([2, 1, 352, 352])
1 torch.Size([2, 1, 352, 352])
2 torch.Size([2, 1, 352, 352])
3 torch.Size([2, 1, 352, 352])
4 torch.Size([2, 3])
5 torch.Size([2, 512])


In [24]:
def structure_loss(pred, mask): #structure_loss = BCE pesata + IoU pesata
    weit = 1 + 5 * torch.abs(
        F.avg_pool2d(mask, kernel_size=31, stride=1, padding=15) - mask
    ) #fa una media locale della mask e poi confronta con mask se per esempio sono in una zona uniforme ottenego che la differenza sarà: 1-1=0, quindi il peso sarà 1, se invece sono nel bordo la differenza varrà per esempio 0.5, quindi peso alto. quindi bordi: peso alto, zone interne: peso basso
    wbce = F.binary_cross_entropy_with_logits(pred, mask, reduction="none")
    wbce = (weit * wbce).sum(dim=(2, 3)) / weit.sum(dim=(2, 3)) #moltiplica errore per peso e fa media pesata

    pred = torch.sigmoid(pred) #ora trasformo logits in probabilità
    inter = ((pred * mask) * weit).sum(dim=(2, 3))
    union = ((pred + mask) * weit).sum(dim=(2, 3))
    wiou = 1 - (inter + 1) / (union - inter + 1) #loss= 1-IoU= 1- (inter/union): qui 1 serve per evitare la divisione per zero

    return (wbce + wiou).mean()

In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TransRaUNet_CLF_xiaorong(training=True).to(device)
cls_ce = nn.CrossEntropyLoss()
cls_reg = SmoothL1Loss()

batch = next(iter(train_loader))
images, gts, labels = batch

images = images.to(device)
gts = gts.to(device)
labels = labels.to(device)

out5, out4, out3, out2, cls_out, features = model(images)

print(out5.shape, out4.shape, out3.shape, out2.shape, cls_out.shape, features.shape)

loss5 = structure_loss(out5, gts)
loss4 = structure_loss(out4, gts)
loss3 = structure_loss(out3, gts)
loss2 = structure_loss(out2, gts)

weight = torch.tensor([0.0, 0.5, 0.5], device=device) #qui metto 0 per non dare pero alla classe "ovary" perchè nel loro codice: label -> one-hot, es: label = 2 → [0, 0, 1] 
loss1 = cls_ce(cls_out, labels)+  cls_reg(cls_out, labels, weight=weight)

loss = loss1 + loss2 + loss3 + loss4 + loss5

print("loss1:", loss1.item())
print("loss2:", loss2.item())
print("loss3:", loss3.item())
print("loss4:", loss4.item())
print("loss5:", loss5.item())
print("total:", loss.item())

torch.Size([8, 1, 352, 352]) torch.Size([8, 1, 352, 352]) torch.Size([8, 1, 352, 352]) torch.Size([8, 1, 352, 352]) torch.Size([8, 3]) torch.Size([8, 512])
loss1: 3.2477810382843018
loss2: 1.4755358695983887
loss3: 1.5281541347503662
loss4: 1.5468931198120117
loss5: 1.5724886655807495
total: 9.370853424072266


In [ ]:
class LitOvaSeg(L.LightningModule):
    def __init__(self, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()

        self.model = TransRaUNet_CLF_xiaorong(training=True)
        self.cls_ce = nn.CrossEntropyLoss()
        self.cls_reg = SmoothL1Loss()

        
        self.val_auc = torchmetrics.AUROC(task="multiclass", num_classes=3)

    def forward(self, x):
        return self.model(x)

    def compute_loss(self, images, gts, labels):
        out5, out4, out3, out2, cls_out, features = self.model(images)

        loss5 = structure_loss(out5, gts)
        loss4 = structure_loss(out4, gts)
        loss3 = structure_loss(out3, gts)
        loss2 = structure_loss(out2, gts)

        weight = torch.tensor([0.0, 0.5, 0.5], device=self.device)
        
        # se SmoothL1Loss crea problemi, lascia solo la CrossEntropy
        loss1 = self.cls_ce(cls_out, labels) + self.cls_reg(cls_out, labels, weight=weight)
        # loss1 = self.cls_ce(cls_out, labels) + self.cls_reg(cls_out, labels, weight=weight)

        loss = loss1 + loss2 + loss3 + loss4 + loss5
        return loss, loss1, loss2, loss3, loss4, loss5, cls_out

    def training_step(self, batch, batch_idx):
        images, gts, labels = batch
        loss, loss1, loss2, loss3, loss4, loss5, cls_out = self.compute_loss(images, gts, labels)


        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)
        self.log("train_cls_loss", loss1, on_step=True, on_epoch=True)
        self.log("train_seg_loss2", loss2, on_step=True, on_epoch=True)
        self.log("train_seg_loss3", loss3, on_step=True, on_epoch=True)
        self.log("train_seg_loss4", loss4, on_step=True, on_epoch=True)
        self.log("train_seg_loss5", loss5, on_step=True, on_epoch=True)

        return loss

    def validation_step(self, batch, batch_idx):
        images, gts, labels = batch
        loss, loss1, loss2, loss3, loss4, loss5, cls_out = self.compute_loss(images, gts, labels)

        
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_cls_loss", loss1)

        probs = torch.softmax(cls_out, dim=1)
        self.val_auc.update(probs, labels)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(
        self.parameters(),
        lr=self.hparams.lr)
        return optimizer
    
    def on_validation_epoch_end(self):
        auc = self.val_auc.compute()

        self.log("val_auc", auc, prog_bar=True)

        self.val_auc.reset()

In [27]:
checkpoint_callback_seg = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
)

In [28]:
def train():

    wandb.init()
    wandb_logger = WandbLogger(
        project="test_ovamta_seg",
        log_model=False
    )

    lr = wandb.config.lr

    model = LitOvaSeg(
        lr=lr
    )

    trainer = L.Trainer(
        max_epochs=2,
        log_every_n_steps=1,
        logger=wandb_logger,
        callbacks=[checkpoint_callback_seg]
    )

    trainer.fit(
        model,
        train_dataloaders=train_loader,
        val_dataloaders=val_loader
    )
    print("Best checkpoint path:", checkpoint_callback_seg.best_model_path)
    wandb.finish()

In [30]:
sweep_config = {
    "method": "grid", #here you are only saying to experiment all the 3 lr
    "metric": {"name": "val_loss", "goal": "minimize"},
    "parameters": {
        "lr": {"values": [1e-2, 1e-3, 1e-4]}
    }
}

In [31]:
sweep_id = wandb.sweep(sweep_config, project="test_ovamta_seg") #It creates a true sweep in the W&B account
print(sweep_id)

Create sweep with ID: yldjsdxl
Sweep URL: https://wandb.ai/sara-tramontana02-/test_ovamta_seg/sweeps/yldjsdxl
yldjsdxl


In [32]:
wandb.agent(sweep_id, function=train, count=3) #with sweep_id you are telling which sweep to use
#in this case 3 runs since I have 3 lr 

wandb: Agent Starting Run: n0lxe1mf with config:
wandb: 	lr: 0.01
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/saratramontana/.netrc.


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/Users/saratramontana/anaconda3/envs/test-seg/lib/python3.12/site-packages/lightning/pytorch/loggers/wandb.py:400: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type                     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ TransRaUNet_CLF_xiaorong │ 29.8 M │ train │     0 │
│ 1 │ cls_ce  │ CrossEntropyLoss         │      0 │ train │     0 │
│ 2 │ cls_reg │ SmoothL1Loss             │      0 │ train │     0 │
│ 3 │ val_auc │ MulticlassAUROC          │      0 │ train │     0 │
└───┴─────────┴──────────────────────────┴────────┴───────┴───────┘

Trainable params: 29.8 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 29.8 M                                                                                               
Total estimated model params size (MB): 119                                                                        
Modules in train mode: 608                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

wandb: WARNING Config item 'lr' was locked by 'sweep' (ignored update).


...

/Users/saratramontana/anaconda3/envs/test-seg/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: 
UserWarning: No negative samples in targets, false positive value should be meaningless. Returning zero tensor in 
false positive score
  warnings.warn(*args, **kwargs)

/Users/saratramontana/anaconda3/envs/test-seg/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/dat
a_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider 
increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=2` reached.


Best checkpoint path: ./test_ovamta_seg/n0lxe1mf/checkpoints/epoch=1-step=28.ckpt


epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁████████████████
train_cls_loss_epoch,█▁
train_cls_loss_step,█▆▄▄▄▂▅▄▂▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁
train_loss_epoch,█▁
train_loss_step,█▆▅▅▅▄▅▄▄▃▃▃▃▂▂▂▂▄▂▁▂▅▂▁▁▂▁▃
train_seg_loss2_epoch,█▁
train_seg_loss2_step,██▇▇▇▇▇▆▇▅▆▅▅▄▃▃▃▆▄▂▃▄▃▂▁▃▁▄
train_seg_loss3_epoch,█▁
train_seg_loss3_step,█▆▆▅▅▅▅▄▅▃▄▄▄▃▂▃▂▅▃▂▃▃▃▂▁▃▁▄
train_seg_loss4_epoch,█▁
+7,...


In [33]:
import glob

paths = glob.glob("**/*.ckpt", recursive=True)
for p in paths:
    print(p)

test_ovamta_seg/s148uznn/checkpoints/epoch=1-step=28.ckpt
test_ovamta_seg/76ydvp2y/checkpoints/epoch=1-step=10.ckpt
test_ovamta_seg/kqkolin1/checkpoints/epoch=6-step=98.ckpt
test_ovamta_seg/uhod15jw/checkpoints/epoch=9-step=140.ckpt
test_ovamta_seg/n0lxe1mf/checkpoints/epoch=1-step=28.ckpt
test_ovamta_seg/84d8x5fh/checkpoints/epoch=5-step=84.ckpt
test_ovamta_seg/r3vxppwq/checkpoints/epoch=1-step=10.ckpt
test_ovamta_diag/rd4fdern/checkpoints/epoch=0-step=5.ckpt
test_ovamta_diag/x6evxkcw/checkpoints/epoch=8-step=126.ckpt
test_ovamta_diag/aogh062w/checkpoints/epoch=2-step=42.ckpt
test_ovamta_diag/36y34a4q/checkpoints/epoch=7-step=112.ckpt
test_ovamta_diag/az6cubwj/checkpoints/epoch=7-step=112.ckpt
test_ovamta_diag/vr8t1j10/checkpoints/epoch=9-step=140.ckpt
